## Exemplo de Agente com NL2SQL

Neste exemplo, implementamos um Agente com capacidade de converter linguagem natural em consultas SQL executáveis. O fluxo contempla desde a compreensão do schema do banco de dados até a geração, validação e execução segura das queries. Durante o desenvolvimento, serão discutidas decisões de prompt, limitações do modelo e estratégias para aumentar precisão e segurança.

### Objetivo Didático

Demonstrar na prática:

- Como estruturar o contexto do banco de dados (schema awareness)
- Como fornecer tabelas, colunas e relacionamentos ao modelo
- Como construir prompts específicos para geração de SQL
- Como restringir o modelo para gerar apenas consultas de leitura
- Como validar a query antes da execução
- Como executar a consulta no banco e tratar erros
- Como retornar resultados estruturados ao usuário
- Como integrar NL2SQL em um fluxo agêntico

#### Pontos de Atenção

- Ambiguidade semântica em perguntas naturais  
- Erros comuns na geração automática de SQL  
- Riscos de segurança (SQL Injection e comandos destrutivos)  
- Limitações do modelo em queries complexas  
- Estratégias para melhorar precisão (few-shot, restrições explícitas, validações adicionais)


## Parte 1 - Estruturando banco e inserindo dados ficticios

### Instalando biblioteca para comunicar com o banco pgvector

In [1]:
! pip install "psycopg[binary]"


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Amanda Machado\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Carregando as variáveis de ambiente (substitua pelas suas no arquivo .env)

In [45]:
from dotenv import load_dotenv
import os

load_dotenv()

# Credencial embedding e LLM
api_key = os.getenv("OCI_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Informações do banco
ip_banco = os.getenv("IP_BANCO")
db_password = os.getenv("DB_PASSWORD")
db_name = "vetores"
db_user="appuser"
db_port=5432

### Teste de conexão no banco

In [46]:
import psycopg
def get_conn():
    return psycopg.connect(
        host=ip_banco,
        dbname=db_name,
        user=db_user,
        password=db_password,
        port=db_port
    )

conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT version();")
print(cur.fetchone())

cur.close()
conn.close()

('PostgreSQL 15.17 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 11.5.0 20240719 (Red Hat 11.5.0-11), 64-bit',)


### Criação de tabelas ficticias para exemplo

In [4]:
conn = get_conn()
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS tickets (
    id SERIAL PRIMARY KEY,
    cliente TEXT,
    prioridade TEXT,
    modulo TEXT,
    status TEXT,
    data_abertura TIMESTAMP,
    resolvido BOOLEAN
);
""")
conn.commit()

cur.close()
conn.close()

print("Banco estruturado com sucesso")

Banco estruturado com sucesso


### Adição de dados ficticios de histórico de tickets

In [47]:
import datetime


tickets_fake = [
    ("AlphaTech Solutions", "P1", "Financeiro", "Aberto", datetime.datetime(2024, 5, 1), False),
    ("AlphaTech Solutions", "P3", "RH", "Fechado", datetime.datetime(2024, 5, 2), True),

    ("NovaEdge Sistemas", "P2", "Estoque", "Aberto", datetime.datetime(2024, 5, 3), False),
    ("NovaEdge Sistemas", "P3", "Vendas", "Fechado", datetime.datetime(2024, 5, 4), True),

    ("BrightCore Ltda", "P3", "RH", "Aberto", datetime.datetime(2024, 5, 5), False),
    ("BrightCore Ltda", "P4", "Estoque", "Fechado", datetime.datetime(2024, 5, 6), True),

    ("SkyBridge Tech", "P2", "Financeiro", "Fechado", datetime.datetime(2024, 5, 7), True),
    ("SkyBridge Tech", "P3", "Financeiro", "Aberto", datetime.datetime(2024, 5, 8), False),

    ("Orion Digital Group", "P4", "Vendas", "Fechado", datetime.datetime(2024, 5, 9), True),
    ("Orion Digital Group", "P3", "Estoque", "Aberto", datetime.datetime(2024, 5, 10), False),

    ("Vertex Dynamics", "P2", "Estoque", "Fechado", datetime.datetime(2024, 5, 11), True),
    ("Vertex Dynamics", "P3", "RH", "Aberto", datetime.datetime(2024, 5, 12), False),

    ("QuantumWorks", "P3", "RH", "Fechado", datetime.datetime(2024, 5, 13), True),
    ("QuantumWorks", "P4", "Vendas", "Aberto", datetime.datetime(2024, 5, 14), False),

    ("BluePeak Solutions", "P1", "Financeiro", "Fechado", datetime.datetime(2024, 5, 15), True),
    ("BluePeak Solutions", "P3", "Estoque", "Aberto", datetime.datetime(2024, 5, 16), False),

    ("LumenSoft Corp", "P2", "Vendas", "Aberto", datetime.datetime(2024, 5, 17), False),
    ("LumenSoft Corp", "P3", "Financeiro", "Fechado", datetime.datetime(2024, 5, 18), True),

    ("AtlasWave Systems", "P4", "Estoque", "Fechado", datetime.datetime(2024, 5, 19), True),
    ("AtlasWave Systems", "P3", "RH", "Aberto", datetime.datetime(2024, 5, 20), False),

    ("Nexora Technologies", "P3", "RH", "Fechado", datetime.datetime(2024, 5, 21), True),
    ("Nexora Technologies", "P2", "Financeiro", "Aberto", datetime.datetime(2024, 5, 22), False),

    ("Hyperion Labs", "P2", "Financeiro", "Fechado", datetime.datetime(2024, 5, 23), True),
    ("Hyperion Labs", "P4", "Vendas", "Aberto", datetime.datetime(2024, 5, 24), False),

    ("StellarOne Solutions", "P3", "Vendas", "Fechado", datetime.datetime(2024, 5, 25), True),
    ("StellarOne Solutions", "P3", "Estoque", "Aberto", datetime.datetime(2024, 5, 26), False),

    ("PrimeAxis Tech", "P4", "Estoque", "Fechado", datetime.datetime(2024, 5, 27), True),
    ("PrimeAxis Tech", "P2", "Financeiro", "Aberto", datetime.datetime(2024, 5, 28), False),

    ("CoreNova Analytics", "P3", "RH", "Fechado", datetime.datetime(2024, 5, 29), True),
    ("CoreNova Analytics", "P1", "Financeiro", "Aberto", datetime.datetime(2024, 5, 30), False),
]
conn = get_conn()
cur = conn.cursor()
for t in tickets_fake:
    cur.execute("""
        INSERT INTO tickets
        (cliente, prioridade, modulo, status, data_abertura, resolvido)
        VALUES (%s,%s,%s,%s,%s,%s)
    """, t)

conn.commit()
cur.close()
conn.close()

### Parte 2 - Criando fluxo de busca de query no banco

In [44]:
!pip install openai tiktoken langchain-core langchain==1.1.0 langchain-openai==1.1.10


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Amanda Machado\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Usando NL2SQL como contexto da LLM

In [14]:
from openai import OpenAI
base_url = "https://inference.generativeai.us-chicago-1.oci.oraclecloud.com/20231130/actions/v1"  # ou sua URL
client_llm = OpenAI(base_url=base_url, api_key=api_key)

def gerar_sql(pergunta):
    schema = """
    Tabela tickets:
    - id (integer)
    - cliente (text)
    - prioridade (text: P1, P2, P3, P4)
    - modulo (text)
    - status (text)
    - data_abertura (timestamp)
    - resolvido (boolean)
    """

    prompt = f"""
    Você é um especialista em SQL para PostgreSQL.
    Gere apenas a query SQL válida, sem explicação.

    {schema}

    Pergunta:
    {pergunta}
    
    Importante:
    - Sua resposta deve ser um SQL executável e apenas isso
    - Elimine qualquer informação adicional, marcadores ou comentários na resposta final
    """

    resposta = client_llm.chat.completions.create(
        model="xai.grok-4-fast-non-reasoning",
        messages=[{"role":"user","content":prompt}]
    )

    return resposta.choices[0].message.content.strip()

### Pipeline de fluxo não agentico

In [15]:
def executar_nl2sql(pergunta):
    sql = gerar_sql(pergunta)
    print("SQL gerado:", sql)
    conn = get_conn()
    cur = conn.cursor()

    cur.execute(sql)
    resultado = cur.fetchall()
    conn.close()
    cur.close()
    return resultado

### Resultado da busca (sem explicação)

In [16]:
executar_nl2sql("Quantos tickets P1 existem?")

SQL gerado: SELECT COUNT(*) FROM tickets WHERE prioridade = 'P1';


[(2,)]

### [opcional]: linkar o metadado diretamente no contexto

In [61]:
def gerar_sql(schema, pergunta):

    prompt = f"""
    Você é um especialista em SQL para PostgreSQL.
    Gere apenas a query SQL válida, sem explicação.

    {schema}

    Pergunta:
    {pergunta}
    
    Regras obrigatórias:
    - Sua resposta deve ser um SQL executável e somente isso
    - Remova qualquer informação adicional, marcadores, markdown e comentários da resposta (inclusive ```sql)
    - Avalie o script SQL antes de gerar a versão final
    """

    resposta = client_llm.chat.completions.create(
        model="xai.grok-4-fast-non-reasoning",
        messages=[{"role":"user","content":prompt}]
    )

    return resposta.choices[0].message.content.strip()



pergunta = "Quantos tickets tem de P1"
conn = get_conn()
cur = conn.cursor()
cur.execute("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'tickets';
""")
schema = cur.fetchall()
sql = gerar_sql(schema, pergunta)
print("SQL gerado:", sql)
cur.execute(sql)
resultado = cur.fetchall()
conn.close()
cur.close()



print(resultado)

SQL gerado: SELECT COUNT(*) FROM tickets WHERE prioridade = 'P1';
[(5,)]


## Parte 3 - Criando um agente de NL2SQL

### Encapsulando execução de query no banco como tool

In [35]:
from langchain.tools import tool

@tool
def tool_executar_nl2sql(sql: str) -> str:
    """ Realiza operações na base de dados de tickets"""
    conn = get_conn()
    cur = conn.cursor()

    cur.execute(sql)
    resultado = cur.fetchall()
    conn.close()
    cur.close()
    return resultado

### Criando o agente com essa tool

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="xai.grok-4-fast-non-reasoning",
    temperature=0,
    max_tokens=500,
    api_key=api_key,
    base_url=base_url
)

tools = [
    tool_executar_nl2sql
]

react_prompt_template = """
    Schema:
    Tabela tickets:
    - id (integer)
    - cliente (text)
    - prioridade (text: P1, P2, P3, P4)
    - modulo (text)
    - status (text)
    - data_abertura (timestamp)
    - resolvido (boolean)

    Pergunta:
    {pergunta}

    Tools:
    {tools}

    IMPORTANT:
    - Sua resposta deve ter como base apenas o contexto fornecido pela tool, nunca adicione informações que não estejam no contexto
    - Caso não tenha contexto relevante, diz que ainda não possui informações sobre isso na base, mas está em constante atualização
    - Sempre converse com o usuário de maneira gentil e educada
    - Ao executar a tool, seu input deve ser um SQL executável que respeite o schema e apenas isso
    - No input da tool, elimine qualquer informação adicional, marcadores ou comentários na resposta final

    Begin!
"""

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=react_prompt_template,
    
)


### [opcional] Extração da resposta final

In [23]:
from langchain_core.messages import AIMessage

def extract_final_output(agent_response):
    final_text = None
    total_tokens = None

    for msg in reversed(agent_response["messages"]):
        if isinstance(msg, AIMessage):

            if msg.content and final_text is None:
                if "Final Answer:" in msg.content:
                    final_text = msg.content.split("Final Answer:",1)[1].strip()
                else:
                    final_text = msg.content.strip()

            if total_tokens is None:
                total_tokens = agent_response["messages"][-1].usage_metadata["total_tokens"]

        if final_text and total_tokens:
            break

    return final_text, total_tokens

### [opcional] Observabilidade das chamadas das tools

In [37]:
import json
from langchain_core.callbacks import BaseCallbackHandler

class CleanAgentCallback(BaseCallbackHandler):
    def on_tool_start(self, serialized, input_str, **kwargs):
        tool_name = serialized.get("name", "unknown_tool")
        print(f"\n TOOL: {tool_name}")
        print(f"   ↳ input: {input_str}")

    def on_tool_end(self, output, **kwargs):
        print("TOOL RESULT (resumo):")

        content = getattr(output, "content", output)

        try:
            data = json.loads(content)
            print("   ↳", data)
            

        except Exception:
            text = str(content)
            print("   ↳", text)


### Teste 1 - Pergunta simples

In [43]:
callback = CleanAgentCallback()

pergunta = "Quantos tickets tem ativos?"

response = agent.invoke(
    {"messages": pergunta},
    config={"callbacks": [callback]}
)

final_answer, total_tokens = extract_final_output(response)

print("\nFinal answer:")
print(final_answer)

print("\nMetadados:")
print(f"total tokens: {total_tokens}")


 TOOL: tool_executar_nl2sql
   ↳ input: {'sql': "SELECT COUNT(*) FROM tickets WHERE status != 'resolvido' OR resolvido = false;"}
TOOL RESULT (resumo):
   ↳ [[4]]

Final answer:
Olá! De acordo com as informações disponíveis na nossa base de dados, há **4 tickets ativos** no momento. Se precisar de mais detalhes ou filtros específicos, é só pedir! 😊

Metadados:
total tokens: 665


### Teste 2 - Pergunta com margem de interpretação

In [44]:
callback = CleanAgentCallback()

pergunta = "Quantos tickets tem resolvidos?"

response = agent.invoke(
    {"messages": pergunta},
    config={"callbacks": [callback]}
)

final_answer, total_tokens = extract_final_output(response)

print("\nFinal answer:")
print(final_answer)

print("\nMetadados:")
print(f"total tokens: {total_tokens}")


 TOOL: tool_executar_nl2sql
   ↳ input: {'sql': 'SELECT COUNT(*) FROM tickets WHERE resolvido = true;'}
TOOL RESULT (resumo):
   ↳ [[2]]

Final answer:
Olá! De acordo com os dados disponíveis na nossa base, há 2 tickets resolvidos. Se precisar de mais detalhes ou informações sobre algum ticket específico, é só avisar! 😊

Metadados:
total tokens: 658


### Teste 3 - Pergunta complexa

In [64]:
callback = CleanAgentCallback()

pergunta = "Qual a proporção de tickets com a maior prioridade comparado aos demais tickets por segmento"

response = agent.invoke(
    {"messages": pergunta},
    config={"callbacks": [callback]}
)

final_answer, total_tokens = extract_final_output(response)

print("\nFinal answer:")
print(final_answer)

print("\nMetadados:")
print(f"total tokens: {total_tokens}")


 TOOL: tool_executar_nl2sql
   ↳ input: {'sql': "SELECT \n    modulo,\n    COUNT(CASE WHEN prioridade = 'P1' THEN 1 END) AS tickets_p1,\n    COUNT(*) AS total_tickets,\n    ROUND(\n        (COUNT(CASE WHEN prioridade = 'P1' THEN 1 END)::DECIMAL / COUNT(*)) * 100, 2\n    ) AS proporcao_p1_percent\nFROM tickets \nGROUP BY modulo \nORDER BY modulo;"}
TOOL RESULT (resumo):
   ↳ [('Estoque', 0, 9, Decimal('0.00')), ('Financeiro', 5, 11, Decimal('45.45')), ('RH', 0, 8, Decimal('0.00')), ('Vendas', 0, 6, Decimal('0.00'))]

Final answer:
Olá! Baseado nas informações disponíveis na nossa base de dados de tickets, aqui está a proporção de tickets com a maior prioridade (P1) em comparação aos demais, segmentada por módulo. A proporção é calculada como a porcentagem de tickets P1 em relação ao total de tickets por módulo:

- **Estoque**: 0 de 9 tickets são P1 (0.00%)
- **Financeiro**: 5 de 11 tickets são P1 (45.45%)
- **RH**: 0 de 8 tickets são P1 (0.00%)
- **Vendas**: 0 de 6 tickets são P1 (0.00